#### Compare model metrics

#### Sanity check to ensure all tables used are available

In [0]:
from pyspark.sql import functions as SQL_FUNCTIONS

METRICS_TABLE = "workspace.bda_taxi.model_metrics_comparison"
LOGREG_PRED_TABLE = "workspace.bda_taxi.model_preds_logreg"
RF_PRED_TABLE = "workspace.bda_taxi.model_preds_rf"

required_tables = [METRICS_TABLE, LOGREG_PRED_TABLE, RF_PRED_TABLE]

missing = []
for table_name in required_tables:
    try:
        spark.sql(f"DESCRIBE TABLE {table_name}")
    except Exception:
        missing.append(table_name)

if missing:
    raise ValueError(
        "Missing required tables. Run the model notebooks first:\n" + "\n".join(missing)
    )

print("All required tables exist.")

In [0]:
from pyspark.sql import functions as SQL_FUNCTIONS

METRICS_TABLE = "workspace.bda_taxi.model_metrics_comparison"

metrics_df = spark.table(METRICS_TABLE)

print("=== Model Performance Comparison ===")
display(
    metrics_df.orderBy(SQL_FUNCTIONS.desc("auc_roc"))
)

# Identify best model by AUC
metrics_sorted = metrics_df.orderBy(SQL_FUNCTIONS.desc("auc_roc")).collect()
best = metrics_sorted[0]
second = metrics_sorted[1] if len(metrics_sorted) > 1 else None

best_model_name = best["model"]
print("Selected best model by AUC.")
print("Best:", best["model"], "AUC:", best["auc_roc"])

if second:
    auc_gap = float(best["auc_roc"]) - float(second["auc_roc"])
    print("Second:", second["model"], "AUC:", second["auc_roc"])
    print("AUC difference (best - second):", round(auc_gap, 6))

#### Compare model distributions

In [0]:
LOGREG_PRED_TABLE = "workspace.bda_taxi.model_preds_logreg"
RF_PRED_TABLE = "workspace.bda_taxi.model_preds_rf"

logreg_preds = spark.table(LOGREG_PRED_TABLE)
rf_preds = spark.table(RF_PRED_TABLE)

preds_all = logreg_preds.unionByName(rf_preds)

print("=== Row counts per model ===")
preds_all.groupBy("model_name").count().show()

#### Compare Confusion Matrix Values (Aggregated)

In [0]:
print("=== Confusion Matrix Summary (Both Models) ===")

confusion_summary = (
    preds_all
    .groupBy("model_name", "label", "prediction")
    .count()
    .orderBy("model_name", "label", "prediction")
)

display(confusion_summary)

#### Precision, Recall and F1

In [0]:
# Convert aggregated confusion counts into TP/FP/TN/FN per model
conf = (
    confusion_summary
    .groupBy("model_name")
    .pivot("label", [0, 1])
    .agg(SQL_FUNCTIONS.sum(SQL_FUNCTIONS.when(SQL_FUNCTIONS.col("prediction") == 1, SQL_FUNCTIONS.col("count"))))
)

# Easier: build TP/FP/TN/FN explicitly
tp = (preds_all.filter((SQL_FUNCTIONS.col("label") == 1) & (SQL_FUNCTIONS.col("prediction") == 1))
      .groupBy("model_name").count().withColumnRenamed("count", "tp"))
fp = (preds_all.filter((SQL_FUNCTIONS.col("label") == 0) & (SQL_FUNCTIONS.col("prediction") == 1))
      .groupBy("model_name").count().withColumnRenamed("count", "fp"))
tn = (preds_all.filter((SQL_FUNCTIONS.col("label") == 0) & (SQL_FUNCTIONS.col("prediction") == 0))
      .groupBy("model_name").count().withColumnRenamed("count", "tn"))
fn = (preds_all.filter((SQL_FUNCTIONS.col("label") == 1) & (SQL_FUNCTIONS.col("prediction") == 0))
      .groupBy("model_name").count().withColumnRenamed("count", "fn"))

summary = tp.join(fp, "model_name", "outer").join(tn, "model_name", "outer").join(fn, "model_name", "outer").fillna(0)

summary = (
    summary
    .withColumn("precision", SQL_FUNCTIONS.col("tp") / (SQL_FUNCTIONS.col("tp") + SQL_FUNCTIONS.col("fp")))
    .withColumn("recall",    SQL_FUNCTIONS.col("tp") / (SQL_FUNCTIONS.col("tp") + SQL_FUNCTIONS.col("fn")))
    .withColumn("f1_calc",   2 * (SQL_FUNCTIONS.col("precision") * SQL_FUNCTIONS.col("recall")) /
                            (SQL_FUNCTIONS.col("precision") + SQL_FUNCTIONS.col("recall")))
)

print("Precision/Recall/F1 from predictions at default threshold (0.5):")
display(summary.orderBy(SQL_FUNCTIONS.desc("f1_calc")))

#### ROC Curve 

*Note: ROC curves are plotted using a random sample to avoid collecting the full dataset to the driver on Databricks Serverless.

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

def plot_roc_curve(predictions_table: str, model_name: str):
    df = spark.table(predictions_table)
    
    # pdf = (
    #     df.select("label", "probability")
    #       .toPandas()
    # )
    # Limit to a sample for plotting, safe for free/serverless, needed on Free Tier of Databricks
    SAMPLE_N = 100000

    pdf = (
        df.select("label", "probability")
        .orderBy(SQL_FUNCTIONS.rand(42))
        .limit(SAMPLE_N)
        .toPandas()
    )

    # Extract probability of positive class (index 1)
    pdf["prob_pos"] = pdf["probability"].apply(lambda v: float(v[1]))

    fpr, tpr, _ = roc_curve(pdf["label"], pdf["prob_pos"])
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, label=f"{model_name} (AUC = {roc_auc:.3f})")

plt.figure(figsize=(7, 6))

plot_roc_curve(LOGREG_PRED_TABLE, "Logistic Regression")
plot_roc_curve(RF_PRED_TABLE, "Random Forest")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()

#### Comparision table

In [0]:
from pyspark.sql import functions as SQL_FUNCTIONS

FINAL_COMPARISON_TABLE = "workspace.bda_taxi.model_comparison_final"

final_row = (
    metrics_df
    .orderBy(SQL_FUNCTIONS.desc("auc_roc"))
    .limit(1)
    .withColumn("selected_by", SQL_FUNCTIONS.lit("auc_roc"))
    .withColumn("selected_ts", SQL_FUNCTIONS.current_timestamp())
)

(
    final_row.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(FINAL_COMPARISON_TABLE)
)

print("Saved final model selection table:", FINAL_COMPARISON_TABLE)
display(spark.table(FINAL_COMPARISON_TABLE))

#### Threshold analysis

In [0]:

from pyspark.sql import functions as SQL_FUNCTIONS
from pyspark.ml.functions import vector_to_array
thresholds = [0.3, 0.5, 0.7]

# Convert probability VectorUDT -> array so we can index it safely
scored_base = (
    preds_all
    .withColumn("prob_array", vector_to_array(SQL_FUNCTIONS.col("probability")))
    .withColumn("prob_pos", SQL_FUNCTIONS.col("prob_array")[1].cast("double"))
    .withColumn("label_int", SQL_FUNCTIONS.col("label").cast("int"))
)

threshold_results = []

for threshold in thresholds:
    scored = scored_base.withColumn(
        "pred_at_threshold",
        SQL_FUNCTIONS.when(SQL_FUNCTIONS.col("prob_pos") >= SQL_FUNCTIONS.lit(threshold), 1).otherwise(0)
    )

    summary = (
        scored
        .groupBy("model_name")
        .agg(
            SQL_FUNCTIONS.count("*").alias("trip_count"),
            SQL_FUNCTIONS.sum(SQL_FUNCTIONS.when((SQL_FUNCTIONS.col("pred_at_threshold") == 1) & (SQL_FUNCTIONS.col("label_int") == 1), 1).otherwise(0)).alias("tp"),
            SQL_FUNCTIONS.sum(SQL_FUNCTIONS.when((SQL_FUNCTIONS.col("pred_at_threshold") == 1) & (SQL_FUNCTIONS.col("label_int") == 0), 1).otherwise(0)).alias("fp"),
            SQL_FUNCTIONS.sum(SQL_FUNCTIONS.when((SQL_FUNCTIONS.col("pred_at_threshold") == 0) & (SQL_FUNCTIONS.col("label_int") == 0), 1).otherwise(0)).alias("tn"),
            SQL_FUNCTIONS.sum(SQL_FUNCTIONS.when((SQL_FUNCTIONS.col("pred_at_threshold") == 0) & (SQL_FUNCTIONS.col("label_int") == 1), 1).otherwise(0)).alias("fn")
        )
        .withColumn("threshold", SQL_FUNCTIONS.lit(float(threshold)))
        .withColumn("precision", SQL_FUNCTIONS.col("tp") / (SQL_FUNCTIONS.col("tp") + SQL_FUNCTIONS.col("fp")))
        .withColumn("recall", SQL_FUNCTIONS.col("tp") / (SQL_FUNCTIONS.col("tp") + SQL_FUNCTIONS.col("fn")))
        .withColumn(
            "f1_at_threshold",
            2 * (SQL_FUNCTIONS.col("precision") * SQL_FUNCTIONS.col("recall")) /
            (SQL_FUNCTIONS.col("precision") + SQL_FUNCTIONS.col("recall"))
        )
    )

    threshold_results.append(summary)

threshold_comparison = threshold_results[0]
for part in threshold_results[1:]:
    threshold_comparison = threshold_comparison.unionByName(part)

display(threshold_comparison.orderBy("threshold", "model_name"))

# Store the data for later use
THRESHOLD_TABLE = "workspace.bda_taxi.model_threshold_metrics"

(
    threshold_comparison.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable(THRESHOLD_TABLE)
)

print("Saved threshold metrics table:", THRESHOLD_TABLE)

#### Select and Refit Best Model

In [0]:
BEST_MODEL_OUTPUT = "workspace.bda_taxi.model_preds_best"

if best_model_name == "LogisticRegression":
    best_preds = spark.table(LOGREG_PRED_TABLE)
else:
    best_preds = spark.table(RF_PRED_TABLE)

(
    best_preds
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(BEST_MODEL_OUTPUT)
)

print(f"Saved best model predictions to {BEST_MODEL_OUTPUT}")